In [1]:
!pip install -q transformers==4.40.2
!pip install -q ranx
!pip install -q pandas tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 110.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 101.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 4.1.0 requires transformers<5.0.0,>=4.41.0, but you have transformers 4.40.2 which is incompatible.
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.3/99.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.2/249.2 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 859.0/859.0 kB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 73.1 MB/s et

In [2]:
# Top 1k candidate docs
!wget https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-passagetest2019-top1000.tsv.gz
!gunzip msmarco-passagetest2019-top1000.tsv.gz

# 43 queries
!wget https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-test2019-queries.tsv.gz
!gunzip msmarco-test2019-queries.tsv.gz

# Qrels
!wget https://trec.nist.gov/data/deep/2019qrels-pass.txt


--2025-05-24 11:19:51--  https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-passagetest2019-top1000.tsv.gz
Resolving msmarco.z22.web.core.windows.net (msmarco.z22.web.core.windows.net)... 20.150.34.1
Connecting to msmarco.z22.web.core.windows.net (msmarco.z22.web.core.windows.net)|20.150.34.1|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26634062 (25M) [application/x-gzip]
Saving to: ‘msmarco-passagetest2019-top1000.tsv.gz’

msmarco-passagetest 100%[===================>]  25.40M  6.51MB/s    in 5.1s    

2025-05-24 11:19:57 (5.03 MB/s) - ‘msmarco-passagetest2019-top1000.tsv.gz’ saved [26634062/26634062]

--2025-05-24 11:19:58--  https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-test2019-queries.tsv.gz
Resolving msmarco.z22.web.core.windows.net (msmarco.z22.web.core.windows.net)... 20.150.34.1
Connecting to msmarco.z22.web.core.windows.net (msmarco.z22.web.core.windows.net)|20.150.34.1|:443... connected.
HTTP request sent, awaiting 

In [9]:
import pandas as pd

# Load queries
queries_df = pd.read_csv("msmarco-test2019-queries.tsv", sep="\t", header=None, names=["query_id", "query"])
query_dict_original = dict(zip(queries_df["query_id"].astype(str), queries_df["query"]))

# Load candidate passages
candidates_df = pd.read_csv("msmarco-passagetest2019-top1000.tsv", sep="\t", header=None,
                            names=["query_id", "passage_id", "query_text", "passage_text"])


In [10]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch, json
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = T5Tokenizer.from_pretrained("t5-base")
model = T5ForConditionalGeneration.from_pretrained("t5-base").to(device)

expanded_queries = {}

for qid, query in tqdm(query_dict_original.items()):
    prompt = f"expanding the query: {query}"
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

    with torch.no_grad():
        output = model.generate(input_ids, max_length=100, do_sample=True, top_p=0.9, temperature=0.7)

    expanded_query = tokenizer.decode(output[0], skip_special_tokens=True)
    expanded_queries[qid] = expanded_query.strip()


# Save to JSON
with open("expanded_queries.json", "w") as f:
    json.dump(expanded_queries, f, indent=2)

print("Saving the Expanded queries in json format.")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
100%|██████████| 200/200 [00:46<00:00,  4.26it/s]

Saving the Expanded queries in json format.


In [16]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import json
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

models = {
    "minilm": "cross-encoder/ms-marco-MiniLM-L-6-v2",
    "tinybert": "cross-encoder/ms-marco-TinyBERT-L-2-v2"
}

def rerank_and_save(query_dict, query_type):
    for model_name, model_path in models.items():
        print(f"\nReranking of the {query_type} queries being done with {model_name}...")

        tokenizer = AutoTokenizer.from_pretrained(model_path)
        model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
        model.eval()

        output_lines = []

        for qid in tqdm(query_dict.keys()):
            query = query_dict[qid]
            docs = candidates_df[candidates_df["query_id"] == int(qid)]

            scores = []
            for _, row in docs.iterrows():
                doc_id = str(row["passage_id"]).strip()
                passage = str(row["passage_text"]).strip()

                inputs = tokenizer(query, passage, truncation=True, padding=True,
                                   return_tensors="pt", max_length=512)
                inputs = {k: v.to(device) for k, v in inputs.items()}

                with torch.no_grad():
                    score = model(**inputs).logits[0].item()

                scores.append((doc_id, score))

            ranked = sorted(scores, key=lambda x: x[1], reverse=True)
            for rank, (doc_id, score) in enumerate(ranked):
                output_lines.append(f"{qid.strip()} Q0 {doc_id} {rank + 1} {score:.4f} {query_type}-{model_name}")

        # Save run file
        run_file = f"ranking_{query_type}_{model_name}.run"
        with open(run_file, "w") as f:
            f.writelines(line + "\n" for line in output_lines)
            for i, line in enumerate(output_lines[:3]):
                print(f"[Check line {i}]: {line} (tokens: {len(line.split())})")

        print(f"Saving the: {run_file}")

with open("expanded_queries.json", "r") as f:
    query_dict_expanded = json.load(f)

rerank_and_save(query_dict_original, "original")
rerank_and_save(query_dict_expanded, "expanded")



Reranking of the original queries being done with minilm...


100%|██████████| 200/200 [18:33<00:00,  5.57s/it]


[Check line 0]: 1108939 Q0 2846462 1 7.8842 original-minilm (tokens: 6)
[Check line 1]: 1108939 Q0 7152561 2 7.1881 original-minilm (tokens: 6)
[Check line 2]: 1108939 Q0 5291621 3 6.8297 original-minilm (tokens: 6)
Saving the: ranking_original_minilm.run

Reranking of the original queries being done with tinybert...


100%|██████████| 200/200 [09:03<00:00,  2.72s/it]


[Check line 0]: 1108939 Q0 8630525 1 7.8171 original-tinybert (tokens: 6)
[Check line 1]: 1108939 Q0 8630519 2 7.8171 original-tinybert (tokens: 6)
[Check line 2]: 1108939 Q0 1667520 3 7.8024 original-tinybert (tokens: 6)
Saving the: ranking_original_tinybert.run

Reranking of the expanded queries being done with minilm...


100%|██████████| 200/200 [18:42<00:00,  5.61s/it]


[Check line 0]: 1108939 Q0 2846462 1 7.8842 expanded-minilm (tokens: 6)
[Check line 1]: 1108939 Q0 7152561 2 7.1881 expanded-minilm (tokens: 6)
[Check line 2]: 1108939 Q0 5291621 3 6.8297 expanded-minilm (tokens: 6)
Saving the: ranking_expanded_minilm.run

Reranking of the expanded queries being done with tinybert...


100%|██████████| 200/200 [09:13<00:00,  2.77s/it]

[Check line 0]: 1108939 Q0 8630525 1 7.8171 expanded-tinybert (tokens: 6)
[Check line 1]: 1108939 Q0 8630519 2 7.8171 expanded-tinybert (tokens: 6)
[Check line 2]: 1108939 Q0 1667520 3 7.8024 expanded-tinybert (tokens: 6)
Saving the: ranking_expanded_tinybert.run


In [17]:
from ranx import Run, Qrels, evaluate

# Fix: use proper space separator
qrels_df = pd.read_csv("2019qrels-pass.txt", sep=r"\s+", header=None, names=["query_id", "Q0", "doc_id", "relevance"])
qrels_df.to_csv("2019qrels-fixed.txt", sep="\t", index=False, header=False)

qrels = Qrels.from_file("2019qrels-fixed.txt", kind="trec")

results = []

for model_name in models:
    run_orig = Run.from_file(f"ranking_original_{model_name}.run", kind="trec")
    run_exp = Run.from_file(f"ranking_expanded_{model_name}.run", kind="trec")

    metrics = ["ndcg@10", "recall@100", "map@1000"]

    res_orig = evaluate(qrels, run_orig, metrics, make_comparable=True)
    res_exp = evaluate(qrels, run_exp, metrics, make_comparable=True)

    results.append({"Model": model_name, "Query Type": "Original", **res_orig})
    results.append({"Model": model_name, "Query Type": "Expanded", **res_exp})


In [18]:
df = pd.DataFrame(results)
print(df)

# Save for report
#df.to_csv("task4_evaluation_results.csv", index=False)

# Optional LaTeX table for report
#print(df.to_latex(index=False, float_format="%.4f"))


      Model Query Type   ndcg@10  recall@100  map@1000
0    minilm   Original  0.735793    0.533204  0.498550
1    minilm   Expanded  0.388427    0.329298  0.291846
2  tinybert   Original  0.698411    0.512776  0.461743
3  tinybert   Expanded  0.359045    0.314906  0.272596
